# Полная сборка пятнашек

Три функции ниже образуют отдельные этапы: `generate_data` создаёт разрешимое поле заданного размера и сохраняет его в TXT с блоками `DATA` и `COST`; `solve_data` читает файл и решает задачу; `check_solution` независимо повторяет каждый ход, пересчитывает стоимость и проверяет итоговое поле.

Внутри `solver.py` одиночная плитка и последние две плитки линии ставятся поиском Дейкстры по позициям нужных плиток и пустоты. Остаток 3×3 или 4×4 решается методом IDA*. Для большой доски минимальная цена каждого локального этапа не означает минимальную суммарную цену.

In [1]:
from pathlib import Path
import random
import sys

# Ноутбук предполагает запуск из каталога pyatnashki_full_latest.
sys.path.insert(0, str(Path.cwd()))
from solver import Result, Task, format_board, load_task, solve, verify


def generate_data(
    rows: int,
    cols: int,
    path: str | Path,
    *,
    steps: int = 50,
    seed: int = 0,
    costs: tuple[int, int, int, int] = (3, 1, 2, 4),
) -> Path:
    """Создать разрешимые DATA/COST заданного размера и сохранить в TXT.

    Перемешивание выполняется законными ходами от цели. Параметры steps
    и seed позволяют регулировать сложность и повторять эксперимент.
    """

    if not (2 <= rows <= 100 and 2 <= cols <= 100):
        raise ValueError('Размеры должны находиться в диапазоне 2..100')
    if steps < 0 or len(costs) != 4 or any(cost < 0 for cost in costs):
        raise ValueError('Неверное число ходов или стоимости U D L R')

    rng = random.Random(seed)
    board = list(range(1, rows * cols)) + [0]
    blank = rows * cols - 1
    previous = -1

    for _ in range(steps):
        row, col = divmod(blank, cols)
        neighbors = [
            nr * cols + nc
            for nr, nc in ((row - 1, col), (row + 1, col), (row, col - 1), (row, col + 1))
            if 0 <= nr < rows and 0 <= nc < cols and nr * cols + nc != previous
        ]
        if not neighbors:
            neighbors = [previous]
        neighbor = rng.choice(neighbors)
        board[blank], board[neighbor] = board[neighbor], 0
        previous, blank = blank, neighbor

    lines = ['DATA', f'SIZE {rows} {cols}']
    for row in range(rows):
        values = board[row * cols:(row + 1) * cols]
        lines.append(' '.join(str(tile) if tile else '=' for tile in values))
    lines.extend(['/', '', 'COST', ' '.join(map(str, costs)), '/'])

    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return output_path


def solve_data(path: str | Path) -> Result:
    """Прочитать сохранённые DATA/COST и вернуть найденную последовательность."""

    return solve(load_task(path))


def check_solution(path: str | Path, result: Result) -> dict:
    """Проверить каждый ход, пересчитать стоимость и подтвердить полную сборку."""

    task = load_task(path)
    verify(task, result)
    goal = tuple(range(1, task.rows * task.cols)) + (0,)
    if result.final_board != goal:
        raise AssertionError('Поле осталось несобранным')

    return {
        'size': (task.rows, task.cols),
        'initial_board': task.board,
        'final_board': result.final_board,
        'cost': result.cost,
        'moves': len(result.order),
    }


## Три демонстрации

В каждом примере сначала создаётся TXT, затем вызывается решатель и отдельно проверяется его результат. Печатаются начальное поле, число ходов, суммарная цена и конечное поле.

In [2]:
examples = (
    (2, 3, 12, 7),
    (21, 21, 70, 11),
    (3, 20, 55, 19),
)

for rows, cols, steps, seed in examples:
    data_path = generate_data(rows, cols, Path('examples') / f'{rows}x{cols}.txt', steps=steps, seed=seed)
    solution = solve_data(data_path)
    checked = check_solution(data_path, solution)

    print(f'\nПоле {rows}×{cols}; файл: {data_path}')
    print('Начальная конфигурация:')
    print(format_board(checked['initial_board'], cols))
    print(f"Ходов: {checked['moves']}; стоимость: {checked['cost']}")
    print('Конечная конфигурация:')
    print(format_board(checked['final_board'], cols))



Поле 2×3; файл: examples\2x3.txt
Начальная конфигурация:
4 1 2
= 5 3
Ходов: 4; стоимость: 8
Конечная конфигурация:
1 2 3
4 5 =

Поле 21×21; файл: examples\21x21.txt
Начальная конфигурация:
  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21
 22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42
 43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63
 64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84
 85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105
106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147
148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168
169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189
190 191 192 193 194 195 196 197 198 199 200 201 202 20

## Неразрешимый угол на поле 7×7

Если все внешние плитки уже стоят на своих местах, а пустота находится в угловом прямоугольнике, разрешимость этого прямоугольника совпадает с разрешимостью всего поля. Законные ходы сохраняют инвариант: чётность перестановки всех клеток (включая пустоту) в сумме с чётностью манхэттенского расстояния пустоты до цели. Обмен двух плиток меняет только первую часть. Поэтому временное использование собранной области не исправляет такую конфигурацию. Минимальной цены её решения не существует.

Ниже создаются два отрицательных теста: всё поле 7×7 собрано, кроме переставленной пары внутри угла 4×4 или 3×3. Проверяем чётность и всего поля, и локально перенумерованного остатка; оба вызова решателя должны отказать до выполнения ходов. Затем проверяем положительные примеры: тот же угол перемешиваем законными ходами и решаем IDA*. В процессе обычной сборки из разрешимого поля отрицательный случай возникнуть не может.

In [3]:
from solver import Solver, UnsolvableError, is_solvable

for side in (4, 3):
    rows = cols = 7
    top = left = 7 - side
    corner = {r * cols + c for r in range(top, rows) for c in range(left, cols)}
    fixed = set(range(rows * cols)) - corner

    # Обмен двух плиток здесь — намеренная порча входа, а не законный ход.
    path = generate_data(rows, cols, Path('examples') / f'7x7_unsolvable_{side}x{side}.txt', steps=0)
    original = load_task(path)
    corrupted = list(original.board)
    first = top * cols + left
    corrupted[first], corrupted[first + 1] = corrupted[first + 1], corrupted[first]
    lines = ['DATA', 'SIZE 7 7', format_board(tuple(corrupted), cols), '/', 'COST', ' '.join(map(str, original.costs)), '/']
    path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    task = load_task(path)

    local_solver = Solver(task)
    local_solver.locked = fixed.copy()
    local_task = local_solver.remainder_task()
    assert all(task.board[p] == original.board[p] for p in fixed)
    assert not is_solvable(task) and not is_solvable(local_task)

    print(f'\nПоле 7×7 с неразрешимым углом {side}×{side}:')
    print(format_board(task.board, cols))
    print('Разрешимость всего поля:', is_solvable(task))
    print('Разрешимость остатка:', is_solvable(local_task))

    try:
        solve_data(path)
    except UnsolvableError as error:
        print('Общий решатель:', error)
    else:
        raise AssertionError('Неразрешимое поле ошибочно принято')

    try:
        local_solver.solve_remainder()
    except UnsolvableError as error:
        print('Проверка перед IDA*:', error)
    else:
        raise AssertionError('Неразрешимый остаток ошибочно принят')
    assert local_solver.order == [] and local_solver.cost == 0
    print('Стоимость решения: не определена, решения нет.')

    # Контроль: только законные ходы в этом же углу сохраняют разрешимость.
    legal = Solver(original)
    legal.locked = fixed.copy()
    for position in (47, 40, 41):
        legal.move(position)
    reachable = Task(rows, cols, tuple(legal.board), original.costs)
    exact = Solver(reachable)
    exact.locked = fixed.copy()
    assert is_solvable(reachable) and is_solvable(exact.remainder_task())
    print(f'\nКонтрольный разрешимый угол {side}×{side}, начальное поле:')
    print(format_board(reachable.board, cols))
    exact.solve_remainder()
    result = Result(tuple(exact.order), exact.cost, tuple(exact.board))
    verify(reachable, result)
    assert result.final_board == original.board
    print(f'IDA*: ходов {len(result.order)}, стоимость {result.cost}')
    print('Конечное поле:')
    print(format_board(result.final_board, cols))



Поле 7×7 с неразрешимым углом 4×4:
 1  2  3  4  5  6  7
 8  9 10 11 12 13 14
15 16 17 18 19 20 21
22 23 24 26 25 27 28
29 30 31 32 33 34 35
36 37 38 39 40 41 42
43 44 45 46 47 48  =
Разрешимость всего поля: False
Разрешимость остатка: False
Общий решатель: Исходная конфигурация неразрешима по чётности: ни одна последовательность законных ходов не приведёт к цели.
Проверка перед IDA*: Остаток неразрешим по чётности. При правильно собранной внешней области всё поле также неразрешимо; временное перемещение собранных плиток не поможет.
Стоимость решения: не определена, решения нет.

Контрольный разрешимый угол 4×4, начальное поле:
 1  2  3  4  5  6  7
 8  9 10 11 12 13 14
15 16 17 18 19 20 21
22 23 24 25 26 27 28
29 30 31 32 33 34 35
36 37 38 39 40 42  =
43 44 45 46 47 41 48
IDA*: ходов 3, стоимость 9
Конечное поле:
 1  2  3  4  5  6  7
 8  9 10 11 12 13 14
15 16 17 18 19 20 21
22 23 24 25 26 27 28
29 30 31 32 33 34 35
36 37 38 39 40 41 42
43 44 45 46 47 48  =

Поле 7×7 с неразрешимым угл